# Week 4 – Pandas & Data Visualization

## Exploratory Data Analysis of Mauna Loa Weekly Atmospheric CO₂

**Internship phase:** Python & Data Science Fundamentals  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook  
**Dataset period:** March 1958 – December 2001  
**Primary objective:** Load a real observational dataset, understand its structure, clean data quality issues, perform statistical analysis, create at least eight visualizations, and document key observations.

> This notebook uses the historical Mauna Loa weekly CO₂ dataset bundled in `data/raw/mauna_loa_co2_weekly.csv`. The original file is retained unchanged. Missing readings are handled in a separate cleaned DataFrame. The dataset is a historical snapshot and is not a current CO₂ monitoring feed.


## 1. Import required libraries

- **Pandas:** tabular data loading, cleaning, grouping and aggregation.
- **NumPy:** numerical calculations and array operations.
- **Matplotlib:** chart creation and customization.
- **Seaborn:** statistical visualizations such as distributions, boxplots and heatmaps.
- **Pathlib / JSON:** reliable project paths and saving outputs.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
import sys

PROJECT_ROOT = Path.cwd()
# Makes the notebook work when launched from the project root or the notebooks folder.
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_FILE = PROJECT_ROOT / 'data' / 'raw' / 'mauna_loa_co2_weekly.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
for folder in [PROCESSED_DIR, FIGURE_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
print('Libraries imported and output folders prepared.')

## 2. Load the dataset

`read_csv()` loads the CSV file. `parse_dates=['date']` converts the date field to a Pandas datetime type at import time, making date-based sorting and grouping reliable.


In [ ]:
df_raw = pd.read_csv(RAW_FILE, parse_dates=['date'])
print('Dataset loaded successfully.')
print('Shape (rows, columns):', df_raw.shape)
display(df_raw.head())

## 3. Understand the data structure

The first inspection checks column names, data types, non-null counts, memory use, and summary statistics. This is important before making any cleaning decisions.


In [ ]:
print('Column names:')
print(df_raw.columns.tolist())
print('\nDataFrame information:')
df_raw.info()
print('\nDescriptive statistics:')
display(df_raw.describe(include='all'))

## 4. Check missing values

The raw data contains missing weekly CO₂ readings. We quantify both the number and percentage of missing values before choosing a treatment method.


In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': df_raw.isna().sum(),
    'missing_percent': (df_raw.isna().mean() * 100).round(2)
})
display(missing_summary)
print('Total missing cells:', int(df_raw.isna().sum().sum()))

## 5. Check and remove duplicate records

The date field identifies the weekly observation. We check exact duplicate rows and duplicate dates. If repeated dates exist, the first record is retained after sorting; the raw file is never overwritten.


In [ ]:
exact_duplicates = int(df_raw.duplicated().sum())
duplicate_dates = int(df_raw['date'].duplicated().sum())
print('Exact duplicate rows:', exact_duplicates)
print('Duplicate dates:', duplicate_dates)

df_work = df_raw.copy()
df_work = df_work.dropna(subset=['date'])
df_work = df_work.drop_duplicates(subset=['date'], keep='first')
df_work = df_work.sort_values('date').reset_index(drop=True)
print('Rows after duplicate/date cleanup:', len(df_work))

## 6. Handle missing values

CO₂ is a continuous weekly time series. For gaps between two valid weekly measurements, **time-based linear interpolation** estimates a value using the dates and neighboring observations. This method is appropriate for this learning exercise because the measurements are ordered in time and the gaps are relatively short. It is an estimate, not a new measurement.

The raw CSV remains unchanged. Only `df_clean` contains interpolated values.


In [ ]:
df_indexed = df_work.set_index('date')
missing_before = int(df_indexed['co2_ppm'].isna().sum())
df_indexed['co2_ppm'] = df_indexed['co2_ppm'].interpolate(method='time', limit_direction='both')

df_clean = df_indexed.reset_index()
missing_after = int(df_clean['co2_ppm'].isna().sum())
print(f'Missing CO2 values before interpolation: {missing_before}')
print(f'Missing CO2 values after interpolation:  {missing_after}')
print('Raw dataset preserved:', int(df_raw['co2_ppm'].isna().sum()), 'missing values still present in df_raw')
assert missing_after == 0, 'Cleaning did not fill all missing CO2 values.'
display(df_clean.head())

## 7. Feature engineering with Pandas datetime functionality

The raw dataset has a date and CO₂ concentration. We derive year, month, month name, quarter, decade, week number, and a 52-week rolling mean. These fields make grouped and seasonal analysis easier.


In [ ]:
df = df_clean.copy()
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()
df['quarter'] = df['date'].dt.quarter
df['decade'] = (df['year'] // 10) * 10
df['week_number'] = df['date'].dt.isocalendar().week.astype(int)
df['rolling_52_week_mean'] = df['co2_ppm'].rolling(window=52, min_periods=26).mean()

display(df.head())
print('New features:', ['year', 'month', 'month_name', 'quarter', 'decade', 'week_number', 'rolling_52_week_mean'])

## 8. Statistical analysis

We calculate the mean, median, standard deviation, minimum, maximum and quartiles. These summaries describe the distribution of the observed CO₂ concentrations. They should be interpreted in the context of a time series with a strong upward trend.


In [ ]:
numeric_summary = df['co2_ppm'].describe()
additional_stats = pd.Series({
    'variance': df['co2_ppm'].var(),
    'range': df['co2_ppm'].max() - df['co2_ppm'].min(),
    'skewness': df['co2_ppm'].skew(),
    'kurtosis': df['co2_ppm'].kurt()
})
display(numeric_summary.to_frame('co2_ppm_summary'))
display(additional_stats.to_frame('additional_statistic'))

## 9. Grouped analysis using `groupby()`

The annual table shows long-term change. The monthly table summarizes recurring calendar-month differences. The decade table summarizes broad changes across decades.


In [ ]:
annual_summary = df.groupby('year', as_index=False).agg(
    annual_mean_ppm=('co2_ppm', 'mean'),
    annual_median_ppm=('co2_ppm', 'median'),
    annual_std_ppm=('co2_ppm', 'std'),
    observations=('co2_ppm', 'size')
)
annual_summary['year_over_year_change_ppm'] = annual_summary['annual_mean_ppm'].diff()

monthly_summary = df.groupby('month', as_index=False).agg(
    mean_co2_ppm=('co2_ppm', 'mean'),
    median_co2_ppm=('co2_ppm', 'median'),
    observations=('co2_ppm', 'size')
)

decade_summary = df.groupby('decade', as_index=False).agg(
    mean_co2_ppm=('co2_ppm', 'mean'),
    minimum_ppm=('co2_ppm', 'min'),
    maximum_ppm=('co2_ppm', 'max'),
    observations=('co2_ppm', 'size')
)
print('Annual summary (first and last five years):')
display(pd.concat([annual_summary.head(), annual_summary.tail()]))
print('Monthly summary:')
display(monthly_summary)
print('Decade summary:')
display(decade_summary)

## 10. Outlier analysis using the IQR method

The Interquartile Range (IQR) is `Q3 - Q1`. Values below `Q1 - 1.5 × IQR` or above `Q3 + 1.5 × IQR` are flagged for review. For this dataset, a long-term rise is expected, so an IQR flag should not automatically be treated as an error or deleted. We report the count and retain valid observations.


In [ ]:
q1 = df['co2_ppm'].quantile(0.25)
q3 = df['co2_ppm'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outlier_mask = (df['co2_ppm'] < lower_bound) | (df['co2_ppm'] > upper_bound)
print(f'Q1: {q1:.2f} ppm | Q3: {q3:.2f} ppm | IQR: {iqr:.2f} ppm')
print(f'Lower bound: {lower_bound:.2f} ppm | Upper bound: {upper_bound:.2f} ppm')
print('IQR-flagged observations:', int(outlier_mask.sum()))
print('Decision: retain the values and investigate them in time context; do not automatically delete them.')

## 11. Data visualization

The internship task asks for **at least eight visualizations**. This project generates twelve charts. The plotting functions are implemented and documented in `src/analysis.py`; the cells below generate each chart and display the saved image.

1. Raw weekly CO₂ trend (gaps remain visible).
2. Missing-value counts.
3. CO₂ distribution histogram.
4. CO₂ distribution by decade (boxplot).
5. Annual mean CO₂ trend.
6. Calendar-month seasonal profile.
7. Weekly readings and 52-week rolling mean.
8. Year-month heatmap.
9. Year-over-year annual mean change.
10. CO₂ versus year scatter plot with trend line.
11. Correlation heatmap for numeric time features.
12. Monthly boxplot.


In [ ]:
from src.analysis import generate_visualizations

figure_paths = generate_visualizations(df, FIGURE_DIR, raw_data=df_raw)
print(f'Created {len(figure_paths)} visualizations:')
for path in figure_paths:
    print('-', path.name)

### 1. Raw weekly trend

This plot shows raw values with missing periods left as gaps, rather than interpolating them before the cleaning step.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '01_raw_weekly_co2_trend.png'), width=850))

### 2. Missing-value counts

This chart makes the number of unavailable CO₂ observations visible before cleaning.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '02_missing_values.png'), width=850))

### 3. Distribution histogram

This chart shows how often each concentration range occurs across the full observation period.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '03_co2_distribution.png'), width=850))

### 4. Boxplot by decade

This compares the distribution across decades and highlights the shift in concentration levels over time.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '04_co2_by_decade_boxplot.png'), width=850))

### 5. Annual mean trend

Annual averages reduce short-term weekly fluctuations and make the long-term direction easier to see.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '05_annual_mean_trend.png'), width=850))

### 6. Monthly seasonal profile

The monthly averages summarize the recurring within-year pattern in the observations.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '06_monthly_seasonal_profile.png'), width=850))

### 7. 52-week rolling mean

The rolling mean smooths week-to-week variation while preserving the long-term trend.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '07_rolling_52_week_mean.png'), width=850))

### 8. Year-month heatmap

The heatmap shows the concentration level for each year and calendar month.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '08_year_month_heatmap.png'), width=850))

### 9. Annual mean change

This plot shows the change in annual average concentration from one year to the next.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '09_annual_mean_change.png'), width=850))

### 10. Year versus concentration

The scatter plot displays individual weekly observations against calendar year and overlays a linear trend for orientation.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '10_year_vs_co2_scatter.png'), width=850))

### 11. Correlation heatmap

This shows pairwise linear correlations among CO₂ and derived time variables; correlation is not causation.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '11_numeric_correlation_heatmap.png'), width=850))

### 12. Monthly boxplot

This compares monthly distributions and their spread, including seasonal differences.

In [ ]:
display(Image(filename=str(FIGURE_DIR / '12_monthly_boxplot.png'), width=850))

## 12. Key observations

Use the results printed in this notebook and the exported report to describe what the charts show. The conclusions below are descriptive; they do not establish the causes of the observed changes.


In [ ]:
first_year = int(df['year'].min())
last_year = int(df['year'].max())
first_mean = annual_summary.iloc[0]['annual_mean_ppm']
last_mean = annual_summary.iloc[-1]['annual_mean_ppm']
change = last_mean - first_mean
change_pct = change / first_mean * 100

print(f'1. The dataset covers {first_year} to {last_year}.')
print(f'2. The raw file had {missing_before} missing weekly CO2 values; interpolation filled them in the cleaned copy.')
print(f'3. The annual mean increased from {first_mean:.2f} ppm in {first_year} to {last_mean:.2f} ppm in {last_year}.')
print(f'4. This is a change of {change:.2f} ppm ({change_pct:.2f}%) between the first and last annual means in this historical file.')
print(f'5. The month-wise summary ranges from {monthly_summary.mean_co2_ppm.min():.2f} to {monthly_summary.mean_co2_ppm.max():.2f} ppm.')
print(f'6. Exact duplicate rows found before cleaning: {exact_duplicates}; duplicate dates: {duplicate_dates}.')
print(f'7. IQR flagged {int(outlier_mask.sum())} observations; these were retained for time-context review.')

## 13. Export cleaned data and analysis tables

The cleaned CSV is the version with interpolated missing readings. Separate CSV files store annual, monthly and decade summaries. The raw source file is not modified.


In [ ]:
df.to_csv(PROCESSED_DIR / 'mauna_loa_co2_weekly_cleaned.csv', index=False)
annual_summary.to_csv(TABLE_DIR / 'annual_summary.csv', index=False)
monthly_summary.to_csv(TABLE_DIR / 'monthly_summary.csv', index=False)
decade_summary.to_csv(TABLE_DIR / 'decade_summary.csv', index=False)
missing_summary.to_csv(TABLE_DIR / 'missing_value_summary.csv')
print('Exports saved:')
for path in sorted(PROCESSED_DIR.glob('*.csv')) + sorted(TABLE_DIR.glob('*.csv')):
    print('-', path.relative_to(PROJECT_ROOT))

## 14. Conclusion

Pandas was used to load, inspect, clean and aggregate a historical time-series dataset. Matplotlib and Seaborn were used to create twelve visualizations. Missing readings were interpolated in the cleaned copy, duplicate records were checked, summary statistics and group-level summaries were produced, and potential outliers were reviewed without automatically deleting valid observations.

**Important limitation:** interpolation estimates missing weekly readings. These estimates must not be described as direct measurements. The dataset ends in December 2001, so it is intended for this Week 4 exercise rather than current climate reporting.
